# Jev against a foundation model on Azure AI Foundry

The same 26 questions, asked of the same sentences, two ways:

- **Jev**, TypeSafe AI's *System One* model, through `jev_classifier`, as in [notebook 01](01_classify_reviews_with_jev.ipynb).
- **A foundation model** on Azure AI Foundry (the reflection deployment that writes the reviews), through a zero-shot [DSPy](https://dspy.ai) program in `dspy_classifier`. Its signature is built from Jev's own question set, so both are asked exactly the same thing, and both see the sentence and its whole review, never the star rating or the customer.

They are compared on **speed**, **cost**, **agreement** with each other, and **accuracy** against what the generator intended (`review_truth`). See issue #19.

Two things to keep in mind when reading the results:

- Jev *measures* its confidences; the foundation model only *reports* them. They fill the same columns, but are not equally trustworthy.
- The foundation model is a reasoning model. Its hidden reasoning is billed as output tokens and dominates its time, and `REASONING_EFFORT` trades one for the other.

Run [notebook 01](01_classify_reviews_with_jev.ipynb) first: this one reuses its dataset.

In [1]:
from pathlib import Path

import polars as pl
from dotenv import load_dotenv

from dspy_classifier import (
    Run,
    accuracy,
    agreement,
    classify_sentences,
    cost,
    disagreements,
    foundry_lm_from_env,
    speed,
    stratified_sample,
)
from dspy_classifier.comparison import load_run, save_run
from jev_classifier import classify_sentences as jev_classify_sentences
from retail_generator.estimate import (
    JEV_SECONDS_PER_SENTENCE,
    JEV_TOKENS_PER_SENTENCE,
    JEV_USD_PER_MILLION_INPUT,
)
from retail_model import read_dataset
from review_wrangler import load_reviews, product_catalogue, split_sentences

load_dotenv(Path.cwd().parent / ".env")
OUTPUT = Path.cwd().parent / "data" / "output" / "comparison"
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(200)

# The comparison: change these and rerun.
SAMPLE_SENTENCES = 300          # whole reviews, stratified by language and stars (300 gives ~370); None for all
CONCURRENCY = 8                 # the same for both classifiers, so the race is fair
REASONING_EFFORT = None         # None for the deployment's default, or "minimal" / "low" / "medium" / "high"
FOUNDRY_PRICES = (None, None)   # USD per million input and output tokens for the deployment, to cost it
RERUN = False                   # False reuses saved runs: rerunning the analysis never pays twice

## 1. The sample

Whole reviews, so a review's intended issue can be scored against everything found in it, stratified by the language it was written in and its star rating.

In [2]:
sentences = split_sentences(load_reviews()).collect()
catalogue = product_catalogue()
tables = read_dataset()
truth = tables["review_truth"]
sample = sentences if SAMPLE_SENTENCES is None else stratified_sample(sentences, SAMPLE_SENTENCES)
print(f"{sample.height} sentences from {sample['review_id'].n_unique()} reviews, of {sentences.height:,} sentences in the dataset")
sample.group_by("text_language").agg(reviews=pl.col("review_id").n_unique(), sentences=pl.len()).sort("sentences", descending=True)

372 sentences from 84 reviews, of 6,781 sentences in the dataset


text_language,reviews,sentences
str,u32,u32
"""english""",51,228
"""portuguese""",5,25
"""chinese""",5,25
"""japanese""",4,23
"""spanish""",5,20
"""swahili""",5,18
"""zulu""",5,18
"""german""",4,15


## 2. What it will cost

Jev's figures are measured. For the foundation model, the prompt is measured by rendering it, but its output tokens cannot be known in advance: a reasoning model decides how long to think. The run measures them.

In [3]:
from dspy_classifier.comparison import prompt_tokens

jev_tokens = sample.height * JEV_TOKENS_PER_SENTENCE
foundry_prompt = prompt_tokens(sample, catalogue)
print(f"Jev:     ~{jev_tokens:,} input tokens, ~${jev_tokens / 1e6 * JEV_USD_PER_MILLION_INPUT:.2f}, "
      f"~{sample.height * JEV_SECONDS_PER_SENTENCE / CONCURRENCY / 60:.1f} min")
print(f"Foundry: ~{foundry_prompt:,} input tokens plus reasoning and answer tokens, measured by the run")

Jev:     ~1,023,000 input tokens, ~$0.04, ~0.2 min
Foundry: ~1,873,924 input tokens plus reasoning and answer tokens, measured by the run


## 3. Run both, live

Neither run uses a cache, so both are timed fresh, on the same sentences, at the same concurrency. Each run is saved to `data/output/comparison/`, and `RERUN = False` reuses it.

In [4]:
async def run(name, classify):
    saved = OUTPUT / f"{name}.parquet"
    if not RERUN and saved.exists():
        return load_run(saved)
    import time
    started = time.perf_counter()
    results = await classify()
    return save_run(Run(name, results, time.perf_counter() - started, CONCURRENCY), saved)

lm = foundry_lm_from_env(reasoning_effort=REASONING_EFFORT)
jev = await run("jev", lambda: jev_classify_sentences(sample, catalogue, concurrency=CONCURRENCY))
foundry = await run("foundry", lambda: classify_sentences(sample, catalogue, lm=lm, concurrency=CONCURRENCY))
print(f"Foundry model: {lm.model}, reasoning effort: {REASONING_EFFORT or 'deployment default'}")

372 sentence(s) to classify, 0 cached.


Classified 372 in 12.1s; 0 failed.
372 sentence(s) to classify with openai/gpt-5.6-terra, 0 cached.


Classified 372 in 224.8s; 0 failed.
Foundry model: openai/gpt-5.6-terra, reasoning effort: deployment default


## 4. Speed

Latency per sentence, throughput at the same concurrency, and how long the whole dataset would take at that rate.

In [5]:
speed([jev, foundry], sentences_in_dataset=sentences.height)

classifier,sentences,failed,p50_ms,p95_ms,concurrency,sentences_per_s,dataset_minutes
str,i64,i64,f64,f64,i64,f64,f64
"""jev""",372,0,255.0,301.0,8,30.72,3.7
"""foundry""",372,0,2844.0,3880.0,8,1.65,68.3


## 5. Cost

Tokens per sentence and dollars, scaled up to 1,000 customers. Jev charges for input tokens only. Set `FOUNDRY_PRICES` to cost the foundation model.

In [6]:
sentences_per_1000 = round(sentences.height * 1000 / tables["customers"].height)
cost([jev, foundry], {"jev": (JEV_USD_PER_MILLION_INPUT, 0.0), "foundry": FOUNDRY_PRICES},
     sentences_per_1000_customers=sentences_per_1000)

classifier,input_tokens,output_tokens,usd_per_sentence,usd_per_1000_customers
str,i64,i64,f64,f64
"""jev""",2863,0,0.00012,0.82
"""foundry""",5927,381,null,null


## 6. Agreement

Per question, on the sentences both answered. Choices agree on the same label, yes/no questions on the same side of the 0.5 threshold, frustration within one level of its 0–4 rubric.

In [7]:
agreement(jev.results, foundry.results)

question,agreement,sentences
str,f64,i32
"""frustration (within 1)""",0.989,372
"""problem_category""",0.911,372
"""language""",1.0,372
"""sentiment""",0.965,372
"""recommendation""",0.952,372
"""churn_risk""",1.0,372
"""safety_concern""",0.987,372
"""suggestion""",0.989,372
"""competitor_mention""",0.968,372


In [8]:
# Where they disagree about the problem
disagreements(jev.results, foundry.results, sample, "problem_category", names=("jev", "foundry"), n=10)

product_name,sentence,jev,foundry
str,str,str,str
"""SmokeRing Premium Lump Charcoal""","""The pieces were mostly a good size with very little dust in the bag, though I found a coup…","""none""","""size_capacity"""
"""TurboGrill Portable Gas""","""Por este precio me siento estafado; no inspira ninguna confianza para transportar ni para …","""price_value""","""safety"""
"""SmokeRing Premium Lump Charcoal""","""I only knocked off one star because there was a little more charcoal dust in the bottom of…","""cleaning_maintenance""","""build_quality"""
"""GrillGuard Heat Resistant Gloves""","""Tentei tirar a grelha quente e quase derrubei tudo porque a camada de silicone escorrega n…","""ease_of_use""","""safety"""
"""WoodChips Hickory Smoking Chips""","""Opened the bag and found a lot of dust, bark pieces, and tiny splinters instead of consist…","""other""","""build_quality"""
"""WoodChips Hickory Smoking Chips""","""The larger pieces that were usable gave decent hickory flavour, but there were not many of…","""performance""","""build_quality"""
"""SmokeRing Premium Lump Charcoal""","""燃焼時間は十分ですが、袋の底に細かい欠片と粉がやや多めに入っていたのが残念です。""","""cleaning_maintenance""","""build_quality"""
"""SmokeRing Premium Lump Charcoal""","""Ngithole izikhwama ezimbalwa zinezicucu ezincane kakhulu ngaphansi, yingakho nginganikanga…","""other""","""build_quality"""
"""SmokeRing Premium Lump Charcoal""","""Kodwa ngokwakhiwa kwawo kanye nekhwalithi yokhuni, ayawenza umsebenzi kahle.""","""performance""","""none"""


In [9]:
# ...and about the sentiment
disagreements(jev.results, foundry.results, sample, "sentiment", names=("jev", "foundry"), n=10)

product_name,sentence,jev,foundry
str,str,str,str
"""SeasonPro BBQ Rub Collection""","""I only wish the Carolina tang rub came in a slightly larger tin, since we went through tha…","""negative""","""mixed"""
"""SmokeRing Premium Lump Charcoal""","""I only knocked off a star because a couple of pieces were a little oversized for my smalle…","""mixed""","""negative"""
"""SmokeRing Premium Lump Charcoal""","""I only knocked off one star because there was a little more charcoal dust in the bottom of…","""mixed""","""negative"""
"""CleanBurn Pellets""","""The bag had a small amount of dust at the bottom, but nothing excessive.""","""mixed""","""neutral"""
"""SmokeRing Premium Lump Charcoal""","""Kodwa ngokwakhiwa kwawo kanye nekhwalithi yokhuni, ayawenza umsebenzi kahle.""","""negative""","""positive"""
"""ChefsPride Stainless Steel Spatula Set""","""I use the thin-edge flipper most often for eggs and smash burgers on my flat top.""","""neutral""","""positive"""
"""WoodChips Hickory Smoking Chips""","""近所のホームセンターよりかなり高かったので、期待して購入しました。""","""neutral""","""mixed"""
"""FlameStarter Natural Fire Lighter""","""包装里如果能多做一层防潮保护就更好了。""","""negative""","""mixed"""
"""WoodChips Hickory Smoking Chips""","""Só não dou uma estrela porque ainda consegui usar um pouco no defumador.""","""mixed""","""negative"""


## 7. Accuracy against the truth

Neither classifier saw `review_truth`. Scored against it:

- **language:** the share of sentences heard in the language their review was written in (Zulu, which neither offers, should be `other`);
- **issue_found:** of the unhappy reviews, the share where the issue the review was written about is among the problems found in it;
- **primary_issue:** of those, the share where it is the problem in the review's most frustrated sentence.

In [10]:
pl.DataFrame([{"classifier": r.name, **accuracy(r.results, sample, truth)} for r in (jev, foundry)])

classifier,language,issue_found,primary_issue,unhappy_reviews
str,f64,f64,f64,i64
"""jev""",1.0,1.0,0.724,29
"""foundry""",1.0,1.0,0.828,29


## 8. What it shows

From the run saved with this notebook: 372 sentences from 84 reviews, both classifiers at concurrency 8, gpt-5.6-terra at its default reasoning effort.

- **Speed is the difference.** Jev's median latency is 255 ms against 2,844 ms for gpt-5.6-terra, and at the same concurrency it gets through about 19 times as many sentences a second (30.7 against 1.65). The whole 6,781-sentence dataset would take Jev under 4 minutes and gpt-5.6-terra over an hour. The foundation model spends its time reasoning: about 380 output tokens a sentence that Jev does not produce.
- **The prompt costs twice as much.** To ask the same questions, the foundation model reads about 5,900 input tokens a sentence against Jev's 2,900, because the questions have to be spelt out as text, and then it is billed for its output. Set `FOUNDRY_PRICES` to see its cost next to Jev's $0.82 per 1,000 customers.
- **The answers are much the same.** The two agree on 91–100% of answers per question. Problem category agrees least (91%): many of the disagreements are borderline sentences that could fairly carry either label, such as charcoal dust as `cleaning_maintenance` or `build_quality`. Both hear every language correctly, Zulu included.
- **Neither is clearly more accurate here.** Both find the intended issue in every unhappy review. gpt-5.6-terra names it as the primary issue more often (83% against 72%), but that is 3 reviews out of 29, too few to call a difference. A larger sample would settle it.

On this task, Jev gives answers of the same quality an order of magnitude faster, with half the input and no output to pay for. Its confidences are also measured by the model, not reported by it.